# LABEQ V16 — CUDA inference
Select a CUDA GPU from Runtime → Change runtime type. Supply your own permitted PEFT Saudi LoRA checkpoint. The web and inference servers bind to loopback. The authenticated notebook preview is for testing during the session.

In [ ]:
!git clone https://github.com/LeenAlaslani/Labeq.git /content/Labeq
%cd /content/Labeq
%pip install -r requirements.txt unsloth 'transformers>=5' peft


Upload a ZIP that contains one response adapter with adapter_config.json and adapter_model.safetensors. Faisal is optional and follows the original weights' noncommercial license.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import torch

assert torch.cuda.is_available(), 'Select a CUDA GPU runtime first.'
uploads = files.upload()
assert len(uploads) == 1, 'Supply one Saudi PEFT adapter ZIP.'
destination = Path('/content/labeq-response-adapter')
destination.mkdir(exist_ok=False)
with ZipFile(BytesIO(next(iter(uploads.values())))) as archive:
    for filename in ('adapter_config.json', 'adapter_model.safetensors'):
        matches = [entry for entry in archive.namelist() if Path(entry).name == filename]
        assert len(matches) == 1, f'Expected one {filename}'
        (destination / filename).write_bytes(archive.read(matches[0]))


In [ ]:
import os, secrets, subprocess, sys
from google.colab import output

key = secrets.token_urlsafe(32)
use_noncommercial_faisal = False  # @param {type:"boolean"}
model_env = dict(os.environ, LABEQ_ADAPTER_DIR=str(destination),
                 LABEQ_API_KEY=key, LABEQ_USE_FAISAL=str(int(use_noncommercial_faisal)),
                 LABEQ_PRELOAD='1')
web_env = dict(os.environ, LABEQ_MODEL_MODE='real', LABEQ_TTS_PROVIDER='browser',
               LABEQ_MODEL_API_URL='http://127.0.0.1:7862', LABEQ_MODEL_API_KEY=key)
server_log = open('/content/labeq-model.log', 'w')
web_log = open('/content/labeq-web.log', 'w')
def serve(module, port, env, log):
    return subprocess.Popen([sys.executable, '-m', 'uvicorn', module, '--host', '127.0.0.1',
                             '--port', str(port), '--limit-concurrency', '16'], env=env, stdout=log,
                            stderr=subprocess.STDOUT)
model_server = serve('labeq_api:app', 7862, model_env, server_log)
web_server = serve('app:app', 8006, web_env, web_log)
output.serve_kernel_port_as_window(8006)


In [ ]:
import requests
print(requests.get('http://127.0.0.1:8006/api/model-health', timeout=10).json())
print('Model and web logs:', server_log.name, web_log.name)


Stop when you finish testing to release the GPU.

In [ ]:
model_server.terminate()
web_server.terminate()
